# Guideline Search Tool Playground

Use this notebook to test the local USSG guideline lookup tool before wiring it into the agent.

This version keeps things simple:
- it resolves a citation-like query to a local USSG section
- it normalizes subsection-style queries like `USSG §2K2.1(b)(1)` to the base section `2K2.1`
- it returns the section title, full text, and a subsection-centered context snippet when applicable

In [5]:
from __future__ import annotations

import json
from pathlib import Path
import sys

import pandas as pd

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from review_pipeline_v1.search_tools import (
    extract_guideline_citation,
    extract_guideline_section_citation,
    lookup_guideline_section,
)

In [11]:
QUERY = "§2T4.1."
GUIDELINE_YEAR = 2024

In [12]:
citation = extract_guideline_citation(QUERY)
section_citation = extract_guideline_section_citation(QUERY)
print(json.dumps({
    "query": QUERY,
    "citation": citation,
    "section_citation": section_citation,
    "guideline_year": GUIDELINE_YEAR,
}, indent=2))

{
  "query": "\u00a72T4.1.",
  "citation": "2T4.1",
  "section_citation": "2T4.1",
  "guideline_year": 2024
}


In [13]:
lookup_result = lookup_guideline_section(
    QUERY,
    year=GUIDELINE_YEAR,
)
print(json.dumps(lookup_result, indent=2))

{
  "result_type": "local_guideline_lookup",
  "query": "\u00a72T4.1.",
  "requested_citation": "2T4.1",
  "section_citation": "2T4.1",
  "requested_subsection": null,
  "year": 2024,
  "title": "\u00a72T4.1. Tax Table TAX LOSS (APPLY THE GREATEST) OFFENSE LEVEL",
  "entry_id": "ussg|2024|2t4-1",
  "source_type": "ussg",
  "blocks": [
    {
      "subheading": null,
      "text": "(A) $2,500 or less\n(B) More than $2,500\n(C) More than $6,500\n(D) More than $15,000\n(E) More than $40,000\n(F) More than $100,000\n(G) More than $250,000\n(H) More than $550,000\n(I) More than $1,500,000\n(J) More than $3,500,000\n(K) More than $9,500,000\n(L) More than $25,000,000\n(M) More than $65,000,000\n(N) More than $150,000,000\n(O) More than $250,000,000\n(P) More than $550,000,000\n36.\nHistorical\nNote\nEffective November 1, 1987. Amended effective November 1, 1989 (amendment 237); November 1, 1993\n(amendment 491); November 1, 2001 (amendment 617); January 25, 2003 (amendment 647); November 1,\

In [14]:
pd.DataFrame([lookup_result])[[column for column in ["query", "requested_citation", "section_citation", "requested_subsection", "title", "text_length", "block_count"] if column in pd.DataFrame([lookup_result]).columns]]

,query,requested_citation,section_citation,requested_subsection,title,text_length,block_count
0,§2T4.1.,2T4.1,2T4.1,None,§2T4.1. Tax Table TAX LOSS (APPLY THE GREATEST...,651,1


In [10]:
print(lookup_result.get("subsection_context") or "")
print()
print(lookup_result.get("text", "")[:3000])



(a) Base Offense Level (Apply the Greatest):
(1) 24, if the offense (A) created a substantial risk of death or serious bod-
ily injury to any person other than a participant in the offense, and
that risk was created knowingly; or (B) involved the destruction or
attempted destruction of a dwelling, an airport, an aircraft, a mass
transportation facility, a mass transportation vehicle, a maritime fa-
cility, a vessel, or a vessel's cargo, a public transportation system, a
state or government facility, an infrastructure facility, or a place of
public use;
(2) 20, if the offense (A) created a substantial risk of death or serious bod-
ily injury to any person other than a participant in the offense; (B) in-
volved the destruction or attempted destruction of a structure other
than (i) a dwelling, or (ii) an airport, an aircraft, a mass transporta-
tion facility, a mass transportation vehicle, a maritime facility, a ves-
sel, or a vessel's cargo, a public transportation system, a state or g